# Agentic RAG with a Grading Loop

*Optional capstone for Block 2.*

**What you will build:** the knowledge agent from 1.7, upgraded with a **self-correcting loop** — it retrieves, **grades** whether the passages are actually relevant, and if not, **rewrites the query and tries again** before answering. This combines RAG (1.7) with LangGraph cycles (2.5).

> **Run it:** open in [Google Colab](https://colab.research.google.com/github/ezponda/ai-agents-course/blob/main/courses/python_code/book/27_agentic_rag.ipynb) (nothing to install), or run locally in Jupyter. Each notebook installs what it needs in its first cell.

In [ ]:
# Setup — LangChain + LangGraph on OpenRouter. Run once.
!pip install -q "langchain>=1.3,<2.0" "langgraph>=1.2,<2.0" "langchain-openai>=1.3,<2.0"

import os, getpass
from langchain_openai import ChatOpenAI

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Paste your OpenRouter API key: ")

MODEL_NAME = "meta-llama/llama-3.3-70b-instruct"
llm = ChatOpenAI(model=MODEL_NAME, base_url="https://openrouter.ai/api/v1",
                 api_key=os.environ["OPENROUTER_API_KEY"])
print("Ready:", MODEL_NAME)
!pip install -q sentence-transformers

## A small knowledge base (as in 1.7)

Local embeddings, no API key.

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")
documents = [
    "ACME return policy: items can be returned within 30 days with a receipt.",
    "ACME ships to the EU and UK; delivery takes 3 to 5 business days.",
    "ACME Pro plan costs 20 euros per month and includes priority support.",
    "Warranty: ACME hardware is covered for 2 years against manufacturing defects.",
]
doc_vectors = embedder.encode(documents, normalize_embeddings=True)

def retrieve(query, k=2):
    q = embedder.encode([query], normalize_embeddings=True)[0]
    top = np.argsort(-(doc_vectors @ q))[:k]
    return [documents[i] for i in top]

## The graph: retrieve -> grade -> (answer | rewrite -> retrieve)

Plain RAG (1.7) retrieves once and hopes the passages are relevant. Agentic RAG adds a **grader** node and a loop: if the retrieved passages don't answer the question, it rewrites the query and retrieves again — capped so it always terminates.

In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    question: str
    query: str
    docs: list
    relevant: bool
    tries: int
    answer: str

def do_retrieve(state: State) -> dict:
    q = state.get("query", state["question"])
    return {"docs": retrieve(q), "tries": state.get("tries", 0) + 1}

def grade(state: State) -> dict:
    r = llm.invoke([{"role": "system", "content":
                     "Do these passages contain enough to answer the question? Reply exactly YES or NO."},
                    {"role": "user", "content": f"Question: {state['question']}\nPassages:\n" + "\n".join(state["docs"])}])
    return {"relevant": r.content.strip().upper().startswith("YES")}

def rewrite(state: State) -> dict:
    r = llm.invoke([{"role": "system", "content": "Rewrite the search query to retrieve better passages. Output only the query."},
                    {"role": "user", "content": state["question"]}])
    return {"query": r.content.strip()}

def answer(state: State) -> dict:
    r = llm.invoke([{"role": "system", "content": "Answer only from the passages. If they don't cover it, say you don't know."},
                    {"role": "user", "content": f"Question: {state['question']}\nPassages:\n" + "\n".join(state["docs"])}])
    return {"answer": r.content}

def route(state: State) -> str:
    if state["relevant"] or state["tries"] >= 2:
        return "answer"
    return "rewrite"

In [ ]:
builder = StateGraph(State)
builder.add_node("retrieve", do_retrieve)
builder.add_node("grade", grade)
builder.add_node("rewrite", rewrite)
builder.add_node("answer", answer)
builder.add_edge(START, "retrieve")
builder.add_edge("retrieve", "grade")
builder.add_conditional_edges("grade", route, {"answer": "answer", "rewrite": "rewrite"})
builder.add_edge("rewrite", "retrieve")     # the loop
builder.add_edge("answer", END)

graph = builder.compile()

In [ ]:
from IPython.display import Image, display
try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    print("Mermaid image unavailable, showing text instead:\n")
    print(graph.get_graph().draw_mermaid())

In [ ]:
out = graph.invoke({"question": "If my laptop breaks after 14 months, am I covered?"})
print("tries:", out["tries"])
print("answer:", out["answer"])

The grader turns one-shot retrieval into a small research loop — the same self-correction idea as reflection (2.5), applied to search. "agentic" RAG is just RAG with a feedback loop around it, and LangGraph makes that loop explicit and safe to run.

---

**What's next:** one more Block 2 skill — in **28** we add **long-term memory** so an agent remembers a user across separate conversations. Then **Block 3** takes an agent to production: deploy it behind a FastAPI endpoint, connect a real front end, and decide — for any task — whether the answer is raw code, PydanticAI, LangGraph, or n8n.